In [20]:
import torch
from torch_geometric.data import Data
import pandas as po

In [4]:
x = torch.tensor([[2,1], [5,6], [3,7], [12,0]], dtype=torch.float)
y = torch.tensor([0, 1, 0, 1], dtype=torch.float)

In [6]:
# The graph connectivity (edge index) should be confined with the COO format, i.e. the first
#    list contains the index of the source nodes,
#    while the index of target nodes is specified in the second list.
edge_index = torch.tensor([[0, 1, 2, 0, 3],
                           [1, 0, 1, 3, 2]], dtype=torch.long)

In [9]:
data = Data(x=x, y=y, edge_index=edge_index)
data

Data(edge_index=[2, 5], x=[4, 2], y=[4])

In [10]:
from torch_geometric.data import InMemoryDataset

In [13]:
class MyOwnDataset(InMemoryDataset):
    def __init__(self, root, transform=None, pre_transform=None):
        super(MyOwnDataset, self).__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0])

    @property
    def raw_file_names(self):
        return ['some_file_1', 'some_file_2', ...]

    @property
    def processed_file_names(self):
        return ['data.pt']

    def download(self):
        # Download to `self.raw_dir`.
        pass

    def process(self):
        # Read data into huge `Data` list.
        data_list = [...]

        if self.pre_filter is not None:
            data_list = [data for data in data_list if self.pre_filter(data)]

        if self.pre_transform is not None:
            data_list = [self.pre_transform(data) for data in data_list]

        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])

In [15]:
from torch_geometric.data import DataLoader
loader = DataLoader(dataset, batch_size=512, shuffle=True)

NameError: name 'dataset' is not defined

In [17]:
from sklearn.preprocessing import LabelEncoder

In [21]:
df = po.read_csv('Task6_Data.csv')

In [22]:
df.head()

,text,label
0,[ link ] shows a way of graphing the exchange ...,0
1,This graph of time versus position is called a...,1
2,[ link ] is a Feynman diagram for the exchange...,0
3,Feynman diagrams are not only a useful tool fo...,0
4,Special names are given to particles that feel...,0


In [23]:
from torch_geometric.data import InMemoryDataset
from tqdm import tqdm_notebook

In [ ]:
class Task6_Dataset(InMemoryDataset):
    def __init__(self, root, transform=None, pre_transform=None):
        super(YooChooseBinaryDataset, self).__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0])

    @property
    def raw_file_names(self):
        return []
    
    @property
    def processed_file_names(self):
        return ['Task6_Data.csv']

    def download(self):
        pass
    
    def process(self):
        
        data_list = []

        # process by session_id
        grouped = df.groupby('session_id')
        for session_id, group in tqdm(grouped):
            sess_item_id = LabelEncoder().fit_transform(group.item_id)
            group = group.reset_index(drop=True)
            group['sess_item_id'] = sess_item_id
            node_features = group.loc[group.session_id==session_id,['sess_item_id','item_id']].sort_values('sess_item_id').item_id.drop_duplicates().values

            node_features = torch.LongTensor(node_features).unsqueeze(1)
            target_nodes = group.sess_item_id.values[1:]
            source_nodes = group.sess_item_id.values[:-1]

            edge_index = torch.tensor([source_nodes, target_nodes], dtype=torch.long)
            x = node_features

            y = torch.FloatTensor([group.label.values[0]])

            data = Data(x=x, edge_index=edge_index, y=y)
            data_list.append(data)
        
        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])

In [24]:
import os.path as osp
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.nn import GCNConv, ChebConv  # noqa

In [26]:
dataset = 'Cora'
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

In [30]:
class Net(torch.nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = GCNConv(dataset.num_features, 16, cached=True)
        self.conv2 = GCNConv(16, dataset.num_classes, cached=True)
        # self.conv1 = ChebConv(data.num_features, 16, K=2)
        # self.conv2 = ChebConv(16, data.num_features, K=2)

        self.reg_params = self.conv1.parameters()
        self.non_reg_params = self.conv2.parameters()

    def forward(self):
        x, edge_index, edge_weight = data.x, data.edge_index, data.edge_attr
        x = F.relu(self.conv1(x, edge_index, edge_weight))
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index, edge_weight)
        return F.log_softmax(x, dim=1)

In [32]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, data = Net().to(device), data.to(device)
optimizer = torch.optim.Adam([
    dict(params=model.reg_params, weight_decay=5e-4),
    dict(params=model.non_reg_params, weight_decay=0)
], lr=0.01)

In [33]:
def train():
    model.train()
    optimizer.zero_grad()
    F.nll_loss(model()[data.train_mask], data.y[data.train_mask]).backward()
    optimizer.step()

In [34]:
def test():
    model.eval()
    logits, accs = model(), []
    for _, mask in data('train_mask', 'val_mask', 'test_mask'):
        pred = logits[mask].max(1)[1]
        acc = pred.eq(data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    return accs

In [35]:
best_val_acc = test_acc = 0
for epoch in range(1, 10):
    train()
    train_acc, val_acc, tmp_test_acc = test()
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        test_acc = tmp_test_acc
    log = 'Epoch: {:03d}, Train: {:.4f}, Val: {:.4f}, Test: {:.4f}'
    print(log.format(epoch, train_acc, best_val_acc, test_acc))

Epoch: 001, Train: 0.7357, Val: 0.4000, Test: 0.4420
Epoch: 002, Train: 0.8357, Val: 0.5060, Test: 0.5360
Epoch: 003, Train: 0.8429, Val: 0.5600, Test: 0.5810
Epoch: 004, Train: 0.8786, Val: 0.6640, Test: 0.6640
Epoch: 005, Train: 0.9071, Val: 0.6980, Test: 0.7040
Epoch: 006, Train: 0.9071, Val: 0.7180, Test: 0.7320
Epoch: 007, Train: 0.9143, Val: 0.7420, Test: 0.7490
Epoch: 008, Train: 0.9286, Val: 0.7560, Test: 0.7660
Epoch: 009, Train: 0.9357, Val: 0.7620, Test: 0.7680
